In [1]:
import sys
sys.path.append("..")
from src.data_splitting import *
from src.feature_engineering import * 

In [2]:
feature_lag_target = [
    'temperature',
    'humidity',
    'surface_radiation',
    'upper_atmospheric_radiation',
    'air_density',
    'wind_velocity',
    'output_hybrid'
]

In [3]:
df = load_dataset("../datasets/processed/dataset_hybrid.csv")
print(df.shape)
df.head()

(8753, 12)


,timestamp,temperature,humidity,surface_radiation,upper_atmospheric_radiation,output_pv,hour,month,air_density,wind_velocity,output_wind,output_hybrid
0,2019-01-01 08:00:00,-19.889,0.001,82.680,209.252,257.690,8,1,1.335,6.178,1203.800,1461.490
1,2019-01-01 09:00:00,-19.004,0.001,149.553,345.604,530.391,9,1,1.337,6.534,1410.415,1940.806
2,2019-01-01 10:00:00,-17.971,0.001,193.625,430.654,871.016,10,1,1.339,6.747,1547.633,2418.649
3,2019-01-01 11:00:00,-17.281,0.001,203.808,458.550,1087.304,11,1,1.340,6.765,1556.072,2643.376
4,2019-01-01 12:00:00,-16.738,0.001,185.193,427.391,1010.161,12,1,1.342,6.606,1456.741,2466.902


In [4]:
train_raw, eval_raw, test_raw = split_dataset(
    df,
    train_ratio=0.85,
    eval_ratio=0.0
)

print_split_summary(
    train_raw,
    eval_raw,
    test_raw
)

Total: 8753
Train: 7440 (85.00%)
eValidation: 0 (0.00%)
Test: 1313 (15.00%)


In [5]:
drop_column = ["output_pv", "output_wind"]

train_benchmark = create_full_lag_dataset(
    df=train_raw,
    max_lag=72, 
    lag_features=feature_lag_target,
    drop_columns=drop_column
)

test_extended = pd.concat([train_raw.iloc[-72:], test_raw], axis=0)
test_benchmark = create_full_lag_dataset(
    df=test_extended,
    max_lag=72, 
    lag_features=feature_lag_target,
    drop_columns=drop_column
)

In [6]:
train_benchmark.to_csv("../datasets/benchmark/data_train_benchmark_lag.csv", index=False)
test_benchmark.to_csv("../datasets/benchmark/data_test_benchmark_lag.csv", index=False)

In [7]:
feature_summary(train_benchmark, "Train Benchmark")
feature_summary(test_benchmark, "Test Benchmark")


=== TRAIN BENCHMARK ===
Rows                : 7368
Total Features      : 514
Original Features   : 10
Lag Features        : 504

=== TEST BENCHMARK ===
Rows                : 1313
Total Features      : 514
Original Features   : 10
Lag Features        : 504
